In [7]:
from pathlib import Path

database_path = (
    Path.cwd().parent
    / "data"
    / "raw"
    / "MIG_Cement_Records.db"
).resolve()

print("Database path:", database_path)
print("Database exists:", database_path.exists())

Database path: C:\Users\User.DESKTOP-775\Documents\mig-cement-demand-forecasting-E\data\raw\MIG_Cement_Records.db
Database exists: True


In [8]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "gdown"
])

0

In [9]:
from pathlib import Path
import gdown

database_path = (
    Path.cwd().parent
    / "data"
    / "raw"
    / "MIG_Cement_Records.db"
).resolve()

database_path.parent.mkdir(parents=True, exist_ok=True)

file_id = "1XhLbZyif1Qd1pfMb5SHL9NC1K3H3gQjg"

downloaded_file = gdown.download(
    id=file_id,
    output=str(database_path),
    quiet=False
)

print("Downloaded file:", downloaded_file)
print("Database exists:", database_path.exists())

if database_path.exists():
    print(
        "Database size:",
        round(database_path.stat().st_size / 1_048_576, 2),
        "MB"
    )

Downloading...
From: https://drive.google.com/uc?id=1XhLbZyif1Qd1pfMb5SHL9NC1K3H3gQjg
To: C:\Users\User.DESKTOP-775\Documents\mig-cement-demand-forecasting-E\data\raw\MIG_Cement_Records.db
100%|██████████| 3.13M/3.13M [00:01<00:00, 2.99MB/s]

Downloaded file: C:\Users\User.DESKTOP-775\Documents\mig-cement-demand-forecasting-E\data\raw\MIG_Cement_Records.db
Database exists: True
Database size: 2.99 MB


In [10]:
import sqlite3
import pandas as pd

connection = sqlite3.connect(
    f"file:{database_path.as_posix()}?mode=ro",
    uri=True
)

operations = pd.read_sql_query(
    "SELECT * FROM Operations;",
    connection
)

sites = pd.read_sql_query(
    "SELECT * FROM Sites;",
    connection
)

cement_types = pd.read_sql_query(
    "SELECT * FROM CementTypes;",
    connection
)

connection.close()

In [11]:
print("Operations:", operations.shape)
print("Sites:", sites.shape)
print("Cement types:", cement_types.shape)

display(operations.head())
display(sites.head())
display(cement_types.head())

Operations: (32880, 11)
Sites: (30, 4)
Cement types: (3, 1)


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448


,site_id,region,silo_capacity,behavior
0,SITE_001,North,448,aggressive
1,SITE_002,South,288,conservative
2,SITE_003,East,314,aggressive
3,SITE_004,South,472,conservative
4,SITE_005,South,230,aggressive


,cement_type
0,CEM_I
1,CEM_II
2,CEM_III


In [12]:
# Show column names, data types, row counts and non-null counts.
operations.info()

<class 'pandas.DataFrame'>
RangeIndex: 32880 entries, 0 to 32879
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      32880 non-null  str    
 1   site_id                   32880 non-null  str    
 2   cement_type               32880 non-null  str    
 3   planned_pour_tonnes       32880 non-null  float64
 4   consumed_tonnes           32880 non-null  float64
 5   opening_inventory_tonnes  32880 non-null  float64
 6   deliveries_tonnes         32880 non-null  float64
 7   closing_inventory_tonnes  32880 non-null  float64
 8   rain_mm                   32880 non-null  float64
 9   avg_temp_c                32880 non-null  float64
 10  silo_capacity             32880 non-null  int64  
dtypes: float64(7), int64(1), str(3)
memory usage: 2.8 MB


In [13]:
# Review the data types separately in a compact format.
operations.dtypes

date                            str
site_id                         str
cement_type                     str
planned_pour_tonnes         float64
consumed_tonnes             float64
opening_inventory_tonnes    float64
deliveries_tonnes           float64
closing_inventory_tonnes    float64
rain_mm                     float64
avg_temp_c                  float64
silo_capacity                 int64
dtype: object

In [14]:
# Convert the date column from SQLite text into pandas datetime.
# Invalid values will be converted to NaT (Not a Time).
operations["date"] = pd.to_datetime(
    operations["date"],
    errors="coerce"
)

# Display the date range covered by the dataset.
print("Start date:", operations["date"].min())
print("End date:", operations["date"].max())

# Count dates that could not be converted successfully.
print("Invalid dates:", operations["date"].isna().sum())

Start date: 2022-01-01 00:00:00
End date: 2024-12-31 00:00:00
Invalid dates: 0


In [15]:
# Generate summary statistics for numerical columns.
# These include count, mean, standard deviation, minimum and maximum.
operations.describe().T

,count,mean,min,25%,50%,75%,max,std
date,32880,2023-07-02 12:00:00,2022-01-01 00:00:00,2022-10-01 18:00:00,2023-07-02 12:00:00,2024-04-01 06:00:00,2024-12-31 00:00:00,NaN
planned_pour_tonnes,32880.0,30.697158,0.0,12.83,33.455,47.6125,69.98,19.493713
consumed_tonnes,32880.0,23.720475,0.0,10.71,19.72,36.4925,69.97,16.846251
opening_inventory_tonnes,32880.0,3066.682205,0.0,1.28,50.71,3298.5225,20646.18,5586.029168
deliveries_tonnes,32880.0,29.293009,0.0,19.3,29.49,39.75,50.0,12.3256
closing_inventory_tonnes,32880.0,3072.254739,0.0,1.27,50.66,3312.0625,20658.87,5593.01591
rain_mm,32880.0,5.00812,0.0,1.44,3.47,6.97,50.0,4.997142
avg_temp_c,32880.0,10.085887,-5.0,3.34,9.99,16.7,35.0,8.519946
silo_capacity,32880.0,317.533333,120.0,230.0,314.0,437.0,487.0,112.813279


In [16]:
# Generate descriptive statistics for the site reference table.
sites.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
site_id,30,30,SITE_001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
region,30,4,East,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN
silo_capacity,30.0,NaN,NaN,NaN,317.533333,114.740106,120.0,237.5,314.0,426.5,487.0
behavior,30,3,aggressive,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
# Count missing values in every Operations column.
missing_count = operations.isna().sum()

# Calculate the percentage of missing values in every column.
missing_percentage = (
    operations.isna().mean() * 100
).round(2)
# Combine the counts and percentages into one profiling table.
missing_values = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})

# Sort columns from the highest to the lowest missing-value percentage.
missing_values = missing_values.sort_values(
    by="missing_percentage",
    ascending=False
)

display(missing_values)

,missing_count,missing_percentage
date,0,0.0
site_id,0,0.0
cement_type,0,0.0
planned_pour_tonnes,0,0.0
consumed_tonnes,0,0.0
opening_inventory_tonnes,0,0.0
deliveries_tonnes,0,0.0
closing_inventory_tonnes,0,0.0
rain_mm,0,0.0
avg_temp_c,0,0.0


In [18]:
# Count completely duplicated rows.
complete_duplicates = operations.duplicated().sum()

print("Completely duplicated rows:", complete_duplicates)

Completely duplicated rows: 0


In [19]:
# Define the expected business key from the data dictionary.
business_key = [
    "date",
    "site_id",
    "cement_type"
]

# Count repeated combinations of date, site and cement type.
duplicate_keys = operations.duplicated(
    subset=business_key,
    keep=False
)

print(
    "Rows with duplicated business keys:",
    duplicate_keys.sum()
)

# Display duplicates if any exist.
display(
    operations.loc[duplicate_keys]
    .sort_values(business_key)
    .head(20)
)

Rows with duplicated business keys: 0


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity


In [20]:
# Display the number of records for each cement type.
print("Operational cement types:")
display(operations["cement_type"].value_counts())

Operational cement types:


cement_type
CEM_II     11097
CEM_I      10913
CEM_III    10870
Name: count, dtype: int64

In [21]:
# Display the number of operational records for each site.
print("Records by site:")
display(
    operations["site_id"]
    .value_counts()
    .sort_index()
)

Records by site:


site_id
SITE_001    1096
SITE_002    1096
SITE_003    1096
SITE_004    1096
SITE_005    1096
SITE_006    1096
SITE_007    1096
SITE_008    1096
SITE_009    1096
SITE_010    1096
SITE_011    1096
SITE_012    1096
SITE_013    1096
SITE_014    1096
SITE_015    1096
SITE_016    1096
SITE_017    1096
SITE_018    1096
SITE_019    1096
SITE_020    1096
SITE_021    1096
SITE_022    1096
SITE_023    1096
SITE_024    1096
SITE_025    1096
SITE_026    1096
SITE_027    1096
SITE_028    1096
SITE_029    1096
SITE_030    1096
Name: count, dtype: int64

In [22]:
# Examine the distribution of site regions.
print("Sites by region:")
display(sites["region"].value_counts())

Sites by region:


region
East     12
South    11
West      4
North     3
Name: count, dtype: int64

In [23]:
# Examine the distribution of site operating behaviours.
print("Sites by behaviour:")
display(sites["behavior"].value_counts())

Sites by behaviour:


behavior
aggressive      14
conservative     9
chaotic          7
Name: count, dtype: int64

In [24]:
# Identify operational records with site IDs absent from the Sites table.
orphan_sites = operations.loc[
    ~operations["site_id"].isin(sites["site_id"])
]

print("Records with unknown site IDs:", len(orphan_sites))

Records with unknown site IDs: 0


In [25]:
# Identify operational records with cement types absent from CementTypes.
valid_cement_types = cement_types["cement_type"]

orphan_cement_types = operations.loc[
    ~operations["cement_type"].isin(valid_cement_types)
]

print(
    "Records with unknown cement types:",
    len(orphan_cement_types)
)

Records with unknown cement types: 0


In [26]:
# Define fields that should not contain negative values.
non_negative_columns = [
    "planned_pour_tonnes",
    "consumed_tonnes",
    "opening_inventory_tonnes",
    "deliveries_tonnes",
    "closing_inventory_tonnes",
    "rain_mm",
    "silo_capacity"
]

# Count negative values in each selected column.
negative_value_summary = pd.Series({
    column: (operations[column] < 0).sum()
    for column in non_negative_columns
}, name="negative_value_count")

display(negative_value_summary.to_frame())

,negative_value_count
planned_pour_tonnes,0
consumed_tonnes,0
opening_inventory_tonnes,0
deliveries_tonnes,0
closing_inventory_tonnes,0
rain_mm,0
silo_capacity,0


In [27]:
# Calculate the expected closing inventory using the business rule:
# closing inventory = opening inventory + deliveries - consumption.
operations["calculated_closing_inventory"] = (
    operations["opening_inventory_tonnes"]
    + operations["deliveries_tonnes"]
    - operations["consumed_tonnes"]
)

# Measure the difference between reported and calculated closing inventory.
operations["inventory_balance_difference"] = (
    operations["closing_inventory_tonnes"]
    - operations["calculated_closing_inventory"]
)

# Allow a tolerance of 0.01 tonnes for rounding differences.
balance_errors = operations.loc[
    operations["inventory_balance_difference"].abs() > 0.01
]

print("Inventory balance errors:", len(balance_errors))

# Display records that fail the inventory-balance validation.
display(balance_errors.head(20))

Inventory balance errors: 2


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,calculated_closing_inventory,inventory_balance_difference
25208,2022-01-01,SITE_024,CEM_II,64.29,64.29,73.23,36.61,45.56,3.11,16.33,290,45.55,0.01
28496,2022-01-01,SITE_027,CEM_II,12.38,12.38,62.79,13.10,63.52,4.44,10.69,314,63.51,0.01


In [28]:
# Identify records where opening inventory exceeds stated silo capacity.
opening_over_capacity = operations[
    operations["opening_inventory_tonnes"]
    > operations["silo_capacity"]
]

# Identify records where closing inventory exceeds stated silo capacity.
closing_over_capacity = operations[
    operations["closing_inventory_tonnes"]
    > operations["silo_capacity"]
]

print(
    "Opening inventory over capacity:",
    len(opening_over_capacity)
)

print(
    "Closing inventory over capacity:",
    len(closing_over_capacity)
)

Opening inventory over capacity: 11427
Closing inventory over capacity: 11439


In [29]:
# Calculate cement available during the day.
operations["available_inventory_tonnes"] = (
    operations["opening_inventory_tonnes"]
    + operations["deliveries_tonnes"]
)

# Calculate the portion of planned demand that was not consumed.
# clip(lower=0) prevents negative unmet-demand values.
operations["unmet_demand_tonnes"] = (
    operations["planned_pour_tonnes"]
    - operations["consumed_tonnes"]
).clip(lower=0)

# Flag records where closing inventory reached zero.
operations["stockout_flag"] = (
    operations["closing_inventory_tonnes"] <= 0
).astype(int)

# Summarize stockouts.
stockout_count = operations["stockout_flag"].sum()
stockout_rate = operations["stockout_flag"].mean() * 100

print("Stockout records:", stockout_count)
print(f"Stockout rate: {stockout_rate:.2f}%")

Stockout records: 7950
Stockout rate: 24.18%


In [30]:
# A pour is considered ready when actual consumption meets or exceeds
# the scheduled planned-pour quantity.
operations["pour_ready_flag"] = (
    operations["consumed_tonnes"]
    >= operations["planned_pour_tonnes"]
).astype(int)

# Calculate the overall percentage of ready pours.
pour_readiness = (
    operations["pour_ready_flag"].mean() * 100
)

print(f"Baseline pour readiness: {pour_readiness:.2f}%")
print("Project target: at least 98.00%")

Baseline pour readiness: 60.31%
Project target: at least 98.00%


In [31]:
# Add region and behaviour to each operational record.
# validate='many_to_one' confirms that many Operations rows map
# to exactly one Sites record.
profile_data = operations.merge(
    sites[["site_id", "region", "behavior"]],
    on="site_id",
    how="left",
    validate="many_to_one"
)

print("Combined dataset shape:", profile_data.shape)

display(profile_data.head())

Combined dataset shape: (32880, 19)


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,calculated_closing_inventory,inventory_balance_difference,available_inventory_tonnes,unmet_demand_tonnes,stockout_flag,pour_ready_flag,region,behavior
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,63.85,0.000000e+00,98.39,8.64,0,0,North,aggressive
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,38.56,7.105427e-15,83.82,0.00,0,1,North,aggressive
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,47.06,0.000000e+00,85.75,0.00,0,1,North,aggressive
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,32.64,0.000000e+00,65.80,0.00,0,1,North,aggressive
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,0.00,0.000000e+00,47.04,9.84,1,0,North,aggressive


In [32]:
# Calculate site-level pour readiness and unmet demand.
site_readiness = (
    profile_data
    .groupby("site_id", as_index=False)
    .agg(
        total_records=("date", "size"),
        ready_pours=("pour_ready_flag", "sum"),
        total_unmet_demand=(
            "unmet_demand_tonnes",
            "sum"
        )
    )
)

# Calculate the pour-readiness percentage for every site.
site_readiness["pour_readiness_percentage"] = (
    site_readiness["ready_pours"]
    / site_readiness["total_records"]
    * 100
).round(2)

# Show the poorest-performing sites first.
site_readiness = site_readiness.sort_values(
    "pour_readiness_percentage"
)

display(site_readiness.head(10))

,site_id,total_records,ready_pours,total_unmet_demand,pour_readiness_percentage
10,SITE_011,1096,346,14863.98,31.57
7,SITE_008,1096,358,14454.45,32.66
20,SITE_021,1096,366,14354.94,33.39
21,SITE_022,1096,369,14418.57,33.67
19,SITE_020,1096,373,14020.69,34.03
16,SITE_017,1096,381,14152.06,34.76
6,SITE_007,1096,387,14536.97,35.31
4,SITE_005,1096,387,13441.99,35.31
0,SITE_001,1096,392,13612.88,35.77
17,SITE_018,1096,397,13487.49,36.22
